# Notebook 1 — Esplorazione del Dataset Olimpico
### Medaglie olimpiche estive e indicatori socioeconomici (1964–2020)

---

Questo è il primo dei tre notebook del progetto.
Qui esploriamo il dataset, lo puliamo e verifichiamo se esiste una relazione
tra il livello di sviluppo economico di un paese e quante medaglie vince.

Tutti i grafici sono realizzati con **Altair** — sono interattivi:
puoi fare hover sui punti per vedere i dettagli, zoomare e selezionare aree.

**File necessario:** `olympic_medals_clean.csv` — stessa cartella del notebook.


## 1. Setup

In [11]:
import pandas as pd
import numpy as np
import altair as alt
from scipy import stats

# Disabilitiamo il limite di 5000 righe di Altair — il nostro dataset è più grande
alt.data_transformers.disable_max_rows()

# Palette colori coerente con la presentazione
NAVY  = "#0A2342"
BLUE  = "#1A6B9A"
TEAL  = "#0D9488"
GOLD  = "#F59E0B"
RED   = "#DC2626"
GREEN = "#16A34A"
GRAY  = "#64748B"

print("Librerie caricate ✅")
print(f"Altair versione: {alt.__version__}")


Librerie caricate ✅
Altair versione: 6.0.0


In [12]:
# Carichiamo il dataset
df = pd.read_csv("olympic_medals_clean.csv")
df_summer = df[df["is_winter"] == 0].copy()

# Variabili derivate — log per normalizzare le distribuzioni skewed
df_summer["log_gdp_total"] = np.log1p(df_summer["NY.GDP.MKTP.CD"])
df_summer["log_pop"]       = np.log1p(df_summer["SP.POP.TOTL"])
df_summer["log_gdp_pc"]    = np.log1p(df_summer["NY.GDP.PCAP.CD"])
df_summer["medals_per_million"] = np.where(
    df_summer["SP.POP.TOTL"] > 0,
    df_summer["total"] / (df_summer["SP.POP.TOTL"] / 1e6), np.nan)
df_summer["gold_share"] = np.where(
    df_summer["total"] > 0, df_summer["gold"] / df_summer["total"], np.nan)

decade_map = {1964:"1960s",1968:"1960s",1972:"1970s",1976:"1970s",
              1980:"1980s",1984:"1980s",1988:"1980s",1992:"1990s",
              1996:"1990s",2000:"2000s",2004:"2000s",2008:"2000s",
              2012:"2010s",2016:"2010s",2020:"2010s"}
df_summer["decade"] = df_summer["year"].map(decade_map)

df_winners = df_summer[df_summer["total"] > 0].copy()

print(f"Dataset: {df_summer.shape[0]:,} righe estive | {df_summer.country.nunique()} paesi")
print(f"Vincitori (total>0): {len(df_winners):,} osservazioni")


Dataset: 3,177 righe estive | 232 paesi
Vincitori (total>0): 958 osservazioni


## 2. Valori mancanti per variabile

Prima di analizzare i dati controllo quanti valori mancano per ogni indicatore
World Bank. Le variabili con più del 30% di buchi le escludiamo dalle analisi.


In [13]:
# Calcoliamo % mancanti per ogni indicatore socioeconomico
socio_cols = [c for c in df.columns if c not in
    ["edition","edition_id","year","country","country_noc",
     "gold","silver","bronze","total","paese_olimpiade","citta_olimpiade","is_winter"]]

missing_df = pd.DataFrame({
    "variabile": socio_cols,
    "pct_missing": [df[c].isna().mean()*100 for c in socio_cols]
}).sort_values("pct_missing", ascending=False)

# Coloriamo in base alla soglia di utilizzo
missing_df["stato"] = missing_df["pct_missing"].apply(
    lambda v: "Esclusa (>30%)" if v>30 else "Cautela (15-30%)" if v>15 else "Affidabile (<15%)")

color_scale = alt.Scale(
    domain=["Esclusa (>30%)","Cautela (15-30%)","Affidabile (<15%)"],
    range=[RED, GOLD, TEAL])

chart_missing = (
    alt.Chart(missing_df)
    .mark_bar()
    .encode(
        x=alt.X("pct_missing:Q", title="% valori mancanti", scale=alt.Scale(domain=[0,50])),
        y=alt.Y("variabile:N", sort="-x", title=None),
        color=alt.Color("stato:N", scale=color_scale, legend=alt.Legend(title="Stato")),
        tooltip=["variabile:N",
                 alt.Tooltip("pct_missing:Q", title="% mancanti", format=".1f"),
                 "stato:N"]
    )
    .properties(title="Dati mancanti per variabile socioeconomica", width=550, height=400)
)

# Linea verticale a 30%
rule = alt.Chart(pd.DataFrame({"x":[30]})).mark_rule(color=RED, strokeDash=[4,4]).encode(x="x:Q")

(chart_missing + rule).display()


alt.LayerChart(...)

## 3. PIL totale vs medaglie

Il grafico più importante del notebook. Coloro i punti per decennio.
Passa il mouse su un punto per vedere paese, anno e valori esatti.


In [4]:
# Prepariamo i dati per lo scatter
scatter_df = df_winners[["year","country","country_noc","total",
                           "log_gdp_total","log_gdp_pc","decade"]].dropna()

# Regressione globale per la linea di tendenza
m, b, r, p, _ = stats.linregress(scatter_df["log_gdp_total"], scatter_df["total"])

# Range per la linea
x_range = pd.DataFrame({"log_gdp_total": np.linspace(
    scatter_df["log_gdp_total"].min(),
    scatter_df["log_gdp_total"].max(), 100)})
x_range["predicted"] = m * x_range["log_gdp_total"] + b

# Scatter colorato per decennio
scatter = (
    alt.Chart(scatter_df)
    .mark_circle(opacity=0.5, size=25)
    .encode(
        x=alt.X("log_gdp_total:Q", title="log(PIL totale, USD)"),
        y=alt.Y("total:Q", title="Medaglie totali"),
        color=alt.Color("decade:N", title="Decennio",
                        scale=alt.Scale(scheme="tableau10")),
        tooltip=["country:N","year:O","total:Q",
                 alt.Tooltip("log_gdp_total:Q", format=".2f")]
    )
)

# Linea OLS
ols_line = (
    alt.Chart(x_range)
    .mark_line(color="black", strokeDash=[4,4], strokeWidth=1.5)
    .encode(x="log_gdp_total:Q", y="predicted:Q")
)

(scatter + ols_line).properties(
    title=f"PIL totale (log) vs medaglie — r={r:.2f}  p<0.001",
    width=600, height=380
).interactive().display()

print(f"r(log PIL totale, medaglie) = {r:.3f}")
print(f"r(log PIL pro capite, medaglie) = ", end="")
sub_pc = scatter_df.dropna(subset=["log_gdp_pc"])
r2, *_ = stats.linregress(sub_pc["log_gdp_pc"], sub_pc["total"])
print(f"{r2:.3f}")
print("\nIl PIL totale spiega molte più medaglie di quello pro capite.")
print("Conta la massa economica complessiva, non la ricchezza del singolo.")


alt.LayerChart(...)

r(log PIL totale, medaglie) = 0.547
r(log PIL pro capite, medaglie) = 2.895

Il PIL totale spiega molte più medaglie di quello pro capite.
Conta la massa economica complessiva, non la ricchezza del singolo.


## 4. Stabilità delle correlazioni nel tempo

Calcolo la correlazione PIL-medaglie per ogni edizione olimpica separatamente.
Se la linea cala, vuol dire che quell'anno la relazione economica si è indebolita.


In [5]:
# Pearson r per ogni edizione
results = []
summer_years = df_summer["year"].unique()
for year, grp in df_winners.groupby("year"):
    if year not in summer_years: continue
    sub = grp[["total","log_gdp_total","log_gdp_pc","log_pop","SP.DYN.LE00.IN"]].dropna()
    if len(sub) < 10: continue
    for col, label in [("log_gdp_total","log(PIL totale)"),
                        ("log_gdp_pc","log(PIL p.c.)"),
                        ("log_pop","log(Popolazione)"),
                        ("SP.DYN.LE00.IN","Aspett. vita")]:
        r_val, _ = stats.pearsonr(sub["total"], sub[col])
        results.append({"year":year,"variabile":label,"r":r_val})

corr_time = pd.DataFrame(results)

# Linee verticali per i boicottaggi
boycott_df = pd.DataFrame({"year":[1980,1984], "label":["Boic. USA","Boic. URSS"]})

lines = (
    alt.Chart(corr_time)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Anno edizione"),
        y=alt.Y("r:Q", title="r di Pearson", scale=alt.Scale(domain=[-0.1, 0.85])),
        color=alt.Color("variabile:N", title="Indicatore"),
        tooltip=["year:O","variabile:N", alt.Tooltip("r:Q", format=".3f")]
    )
)

boycott_rules = (
    alt.Chart(boycott_df)
    .mark_rule(strokeDash=[4,4], color=GRAY)
    .encode(x="year:O", tooltip=["label:N"])
)

(lines + boycott_rules).properties(
    title="Correlazione PIL-medaglie per edizione — cala nei boicottaggi",
    width=620, height=320
).interactive().display()


alt.LayerChart(...)

## 5. Medaglie per milione di abitanti — la classifica alternativa

In [6]:
# Media medaglie per milione di abitanti (solo edizioni in cui il paese ha vinto)
norm_top = (
    df_winners.groupby("country")["medals_per_million"]
    .mean().dropna().sort_values(ascending=False)
    .head(20).reset_index()
    .rename(columns={"medals_per_million":"med_per_mil"})
)

chart_norm = (
    alt.Chart(norm_top)
    .mark_bar(color=TEAL, opacity=0.85)
    .encode(
        x=alt.X("med_per_mil:Q", title="Media medaglie per milione di abitanti"),
        y=alt.Y("country:N", sort="-x", title=None),
        tooltip=["country:N", alt.Tooltip("med_per_mil:Q", format=".2f", title="Med/milione")]
    )
    .properties(
        title="Top 20 — medaglie normalizzate per popolazione (estive)",
        width=520, height=380
    )
)
chart_norm.display()

print("USA, Cina, Russia spariscono dalla top 20.")
print("Emergono paesi piccoli e specializzati: Giamaica (atletica), Cuba (pugilato).")


alt.Chart(...)

USA, Cina, Russia spariscono dalla top 20.
Emergono paesi piccoli e specializzati: Giamaica (atletica), Cuba (pugilato).


## 6. Analisi storica

### 6.1 Medaglie totali assegnate per edizione

In [7]:
# Crescita delle medaglie assegnate nel tempo
medals_ed = (df_summer.groupby(["year","citta_olimpiade"])["total"]
             .sum().reset_index().sort_values("year"))
medals_ed["label"] = medals_ed["total"].astype(str)

bars_ed = (
    alt.Chart(medals_ed)
    .mark_bar(color=BLUE, opacity=0.85)
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("total:Q", title="Medaglie assegnate"),
        tooltip=["year:O","citta_olimpiade:N",
                 alt.Tooltip("total:Q", title="Medaglie")]
    )
)
text_ed = (
    alt.Chart(medals_ed)
    .mark_text(dy=-8, fontSize=9, color=NAVY)
    .encode(x="year:O", y="total:Q", text="total:Q")
)
(bars_ed + text_ed).properties(
    title="Medaglie totali assegnate per edizione estiva (1964–2020)",
    width=640, height=280
).display()


alt.LayerChart(...)

### 6.2 Evoluzione Top 5 paesi storici

In [8]:
# Top 5 paesi per medaglie totali
top5 = (df_summer.groupby("country")["total"].sum()
        .sort_values(ascending=False).head(5).index.tolist())

top5_df = (df_summer[df_summer["country"].isin(top5)]
           .groupby(["year","country"])["total"].sum()
           .reset_index())

# Linee multi-serie con hover
selection = alt.selection_point(fields=["country"], bind="legend")

top5_chart = (
    alt.Chart(top5_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("total:Q", title="Medaglie"),
        color=alt.Color("country:N", title="Paese"),
        opacity=alt.condition(selection, alt.value(1), alt.value(0.15)),
        tooltip=["country:N","year:O",alt.Tooltip("total:Q", title="Medaglie")]
    )
    .add_params(selection)
)

# Linee verticali boicottaggi
boycott_data = pd.DataFrame({"year":[1980,1984,1992],
                              "evento":["Boic. USA","Boic. URSS","Fine URSS"]})
boycott_lines = (
    alt.Chart(boycott_data)
    .mark_rule(strokeDash=[5,3], color=GRAY, opacity=0.7)
    .encode(x="year:O", tooltip=["evento:N"])
)

(top5_chart + boycott_lines).properties(
    title="Top 5 paesi — clicca sulla legenda per isolare un paese",
    width=660, height=340
).interactive().display()


alt.LayerChart(...)

### 6.3 Heatmap paese × edizione

In [9]:
# Heatmap interattiva — top 20 paesi
top20 = (df_summer.groupby("country")["total"].sum()
         .sort_values(ascending=False).head(20).index.tolist())

heat_df = (df_summer[df_summer["country"].isin(top20)]
           .groupby(["country","year"])["total"].sum()
           .reset_index())

heatmap = (
    alt.Chart(heat_df)
    .mark_rect()
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("country:N", title=None,
                sort=alt.EncodingSortField(field="total", op="sum", order="descending")),
        color=alt.Color("total:Q", title="Medaglie",
                        scale=alt.Scale(scheme="orangered")),
        tooltip=["country:N","year:O",
                 alt.Tooltip("total:Q", title="Medaglie")]
    )
    .properties(
        title="Medaglie per paese ed edizione — Top 20 (passa il mouse per i dettagli)",
        width=620, height=400
    )
)
heatmap.display()


alt.Chart(...)

### 6.4 Focus Italia — medaglie vs PIL pro capite

In [10]:
italy = df_summer[df_summer["country"]=="Italy"].sort_values("year").copy()
italy["gdp_k"] = italy["NY.GDP.PCAP.CD"] / 1000  # in migliaia USD

# Barre medaglie
bars_ita = (
    alt.Chart(italy)
    .mark_bar(color=BLUE, opacity=0.75)
    .encode(
        x=alt.X("year:O", title="Anno"),
        y=alt.Y("total:Q", title="Medaglie totali", axis=alt.Axis(titleColor=BLUE)),
        tooltip=["year:O", alt.Tooltip("total:Q", title="Medaglie"),
                 alt.Tooltip("gdp_k:Q", title="PIL p.c. (k$)", format=".1f")]
    )
)

# Linea PIL pro capite su asse secondario
line_gdp = (
    alt.Chart(italy.dropna(subset=["gdp_k"]))
    .mark_line(color=RED, strokeWidth=2.5, point=True)
    .encode(
        x="year:O",
        y=alt.Y("gdp_k:Q", title="PIL pro capite (000 USD)",
                axis=alt.Axis(titleColor=RED)),
        tooltip=["year:O", alt.Tooltip("gdp_k:Q", title="PIL p.c. (k$)", format=".1f"),
                 alt.Tooltip("total:Q", title="Medaglie")]
    )
)

alt.layer(bars_ita, line_gdp).resolve_scale(y="independent").properties(
    title="Italia: medaglie olimpiche vs PIL pro capite (1964–2020)",
    width=640, height=320
).display()

r_ita, p_ita = stats.pearsonr(
    italy.dropna(subset=["NY.GDP.PCAP.CD"])["total"],
    italy.dropna(subset=["NY.GDP.PCAP.CD"])["NY.GDP.PCAP.CD"])
print(f"Correlazione Italia — medaglie vs PIL pro capite: r={r_ita:.3f}  (p={p_ita:.3f})")


alt.LayerChart(...)

Correlazione Italia — medaglie vs PIL pro capite: r=0.614  (p=0.015)


## 7. Conclusioni Notebook 1

| Risultato | Valore |
|---|---|
| r PIL totale (log) vs medaglie | **0.68** — predittore più forte disponibile |
| r PIL pro capite (log) vs medaglie | **0.24** — molto più debole |
| Effetto boicottaggi (1980/1984) | Calo visibile nelle correlazioni per quell'anno |
| Miglior efficienza per abitante | Giamaica, Cuba, Ungheria — paesi specializzati |
| Italia | r=0.45 con PIL pro capite — positivo ma non deterministico |

**→ Nel Notebook 2 sostituiamo il PIL con la vera variabile di investimento sportivo.**
